# 00 · Diseñar un laboratorio de RAG trazable

**Objetivo:** convertir una pregunta sobre radiología en un experimento auditable:
¿qué documento y versión respaldan una afirmación y qué componente puede fallar?
Este cuaderno es independiente de 01. No requiere clave, red ni modelos descargados.

Primera iteración: preparar el corpus, los contratos y el método de comparación.
No hay todavía retrieval ni respuestas radiológicas generadas. Se espera experiencia
previa en IA; nos centramos en decisiones experimentales y trazabilidad.

**Duración orientativa de estudio:** 25–40 minutos, incluyendo ejercicios.

## 1. Fundamento: RAG como cadena de evidencia

Para una consulta $q$, un recuperador obtiene $D_k = R(q, C)$ y el generador produce
$a = G(q, D_k)$. La calidad final depende de al menos cuatro eventos distintos:
la evidencia existe en el corpus; el parser la conserva; el recuperador la encuentra;
el generador la usa sin añadir afirmaciones no sustentadas. Una respuesta fluida
no identifica cuál de esos eventos ocurrió.

La arquitectura inicial será:

```text
fuente oficial + derechos → original inmutable + SHA-256
                          → páginas / estructura / tablas (01–02)
                          → chunks + procedencia (02)
                          → embeddings locales / retrieval (03)
pregunta → evidencia recuperada → prompt + Ollama Cloud (04) → respuesta y citas
                          ↘ evaluación por componente
```

**RAG frente a documento completo:** con un corpus pequeño, todo el documento podría
caber en contexto. Es una alternativa experimental legítima: comparar ambos bajo el
mismo modelo y preguntas, registrando longitud, ventana disponible, coste y citas.
No asumir que más componentes mejoran la calidad.

**Grafo y KAG:** KG-RAG es una familia; Microsoft GraphRAG, LightRAG y OpenSPG/KAG
son propuestas concretas con pipelines diferentes. Se investigarán cuando existan
fallos relacionales que lo justifiquen. No forman parte de esta instalación.

In [ ]:
import json
import platform
from datetime import UTC, datetime
from importlib.metadata import version

import pandas as pd
from IPython.display import Markdown, display
from radiology_rag.config import Settings
from radiology_rag.ingestion.catalog import load_catalog, sha256_file
from radiology_rag.paths import project_root, within

ROOT = project_root()
catalog = load_catalog(ROOT / "data/catalog/documents.v1.json")
environment = {
    "python": platform.python_version(),
    **{p: version(p) for p in ("pypdf", "pdfplumber", "httpx", "pandas", "nbclient")},
}
display(pd.Series(environment, name="versión ejecutada"))
print("Raíz:", ROOT)
print("Catálogo:", catalog["catalog_version"])

## 2. El corpus real y su estado de adquisición

PI-RADS v2.1 (2019) es el manual enlazado por ACR al revisar el portal el 2026-11-11.
El 2026-11-13 el usuario aportó el PDF de 76 páginas y confirmó la cobertura para
su incorporación y procesamiento. Es el documento activo del catálogo. Se conserva
el original con hash y un manifiesto de procedencia local; no se convierte en fuente abierta.

ESUR-2012 sigue disponible con licencia no comercial explícita como comparación
histórica. Sus criterios no se atribuyen a v2.1. Los otros PDFs siguen pendientes.

Fuentes: [ACR](https://www.acr.org/Clinical-Resources/Clinical-Tools-and-Reference/Reporting-and-Data-Systems/PI-RADS),
[Barentsz et al.](https://doi.org/10.1007/s00330-011-2377-y).


In [ ]:
rows = []
for doc in catalog["documents"]:
    path = within(ROOT, doc["local_path"])
    manifest_path = path.with_suffix(".source.json")
    intact = None
    if path.exists() and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        intact = sha256_file(path) == manifest["sha256"]
    rows.append({
        "id": doc["id"], "versión": doc["version"], "papel": doc["role"],
        "descarga_permitida_sin_permiso_adicional": doc["rights"]["auto_download"],
        "local": path.is_file(), "hash_verificado": intact,
    })
inventory = pd.DataFrame(rows)
display(inventory)
assert not (inventory["hash_verificado"] == False).any(), "Hay fuentes modificadas"

## 3. Experimento de trazabilidad: ¿puedo reconstruir una referencia?

Una referencia debe sobrevivir a cambios de chunking. El ID de chunk no basta:
guardar ID de documento, versión, hash y página física. La sección se anotará tras
revisar estructura, sin inferirla del número de página. El siguiente ejercicio
construye un localizador real del primer archivo disponible, sin inventar citas de contenido.

In [ ]:
available = [d for d in catalog["documents"] if within(ROOT, d["local_path"]).is_file()]
if available:
    doc = available[0]
    source_path = within(ROOT, doc["local_path"])
    reference = {
        "document_id": doc["id"], "version": doc["version"],
        "sha256": sha256_file(source_path), "page_number": 1,
        "section": None, "source_url": doc["source_url"],
    }
    display(pd.Series(reference, name="localizador de página; no cita de una afirmación"))
else:
    print("Sin corpus local. Ejecutar: uv run radiology-rag download esur-2012")
    reference = None

## 4. Separar los tres modelos y las credenciales

El generador consume contexto y produce texto; el encoder representa pasajes/consultas;
el reranker puntúa pares. Sus ventanas, licencias, costes y funciones son diferentes.
En 03 empezaremos con E5 multilingüe local y contrastaremos una alternativa reciente.
La selección se medirá con preguntas ES/EN; aún no hay pesos ni índices.

La configuración se carga desde `.env` de esta raíz y variables del proceso (éstas
tienen precedencia). No mostrar objetos ni volcar el entorno completo: sólo estado
de presencia de clave. Esta celda **no llama a Ollama**.

In [ ]:
settings = Settings.from_env(ROOT / ".env")
display(pd.Series({
    "proveedor_generación": "Ollama Cloud API nativa",
    "origen_configurado": settings.base_url,
    "modelo_configurado": settings.model or "pendiente de selección en /api/tags",
    "clave_presente": bool(settings.api_key),
    "temperatura": settings.temperature,
    "máximo_salida": settings.num_predict,
    "timeout_s": settings.timeout_seconds,
    "reintentos_máximos": settings.max_retries,
}))

La llamada real se ejecuta sólo con `uv run radiology-rag llm-smoke`, después de
configurar un modelo devuelto por `uv run radiology-rag models`. Es una prueba
de conectividad que puede consumir cuota, no una evaluación de RAG. Sin clave no se
simula una respuesta. El comando no envía PDFs.

[API Cloud](https://docs.ollama.com/cloud) · [contrato chat](https://docs.ollama.com/api/chat).

## 5. Diseñar una comparación que no confunda causas

**Unidad experimental:** pregunta + versión documental + corpus identificado por
hash + configuración. Cambiar chunking y modelo a la vez impide atribuir diferencias.

Para preguntas con evidencia relevante $E_q$ y resultados recuperados $D_k$:

$$Recall@k = |E_q \cap D_k| / |E_q|, \qquad Precision@k = |E_q \cap D_k| / k.$$

Esta formulación requiere definir la unidad de relevancia. Si se cambia chunking,
conviene anotar spans/páginas de evidencia y mapearlos a cada estrategia; un gold de
IDs de chunks de un único chunker favorecería ese chunker. La latencia de recuperación
debe separarse de generación. La fidelidad se revisa por afirmación y evidencia.

La tabla siguiente es un **plan**, sin resultados numéricos fabricados.

In [ ]:
experiments = pd.DataFrame([
    {"fase": "01", "variable": "parser y orden de lectura",
     "control": "mismo PDF/páginas", "medición": "perfil, desacuerdo y revisión visual",
     "estado": "ejecutable ahora"},
    {"fase": "02", "variable": "tamaño/overlap/estructura",
     "control": "mismo texto y regiones revisadas", "medición": "evidencia fragmentada, pérdida, duplicación",
     "estado": "pendiente"},
    {"fase": "03", "variable": "encoder/idioma",
     "control": "mismos chunks y preguntas", "medición": "Recall@k, MRR, RAM, latencia",
     "estado": "pendiente"},
    {"fase": "04", "variable": "RAG/documento completo",
     "control": "mismo generador y preguntas", "medición": "citas, fidelidad, abstención, tokens",
     "estado": "pendiente"},
])
display(experiments)

## 6. Interpretación y ejercicios

Mirar el inventario anterior: ¿cuántos documentos son locales y cuántos requieren
autorización? Si sólo existe ESUR-2012, el experimento de v2.1 sigue pendiente.
La tabla de configuración prueba carga de parámetros, no disponibilidad de modelo.

1. Explica qué invalida un experimento si se sustituye un PDF manteniendo el nombre.
2. Propón una pregunta no respondible y una fuera de dominio. No etiquetes respuestas
   hasta identificar documento, versión y evidencia revisada.
3. Diseña un caso en que las palabras de una tabla estén presentes pero la relación
   entre criterio y categoría se haya perdido. ¿Qué métrica de 01 no lo detectaría?
4. Define el presupuesto de contexto completo: tokens de entrada + sistema + pregunta
   + margen de salida ≤ ventana efectiva del modelo, medida con su tokenizer.

In [ ]:
run = {
    "notebook": "00_project_overview",
    "created_at": datetime.now(UTC).isoformat(),
    "catalog_version": catalog["catalog_version"],
    "environment": environment,
    "local_document_ids": inventory.loc[inventory["local"], "id"].tolist(),
    "llm_call_performed": False,
    "reference_example": reference,
}
run_dir = ROOT / "data/processed/overview"
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "run.json").write_text(
    json.dumps(run, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Registro experimental local:", run_dir / "run.json")

## 7. Conclusiones de esta ejecución

Hemos inspeccionado un catálogo real, el entorno y la presencia/integridad del corpus.
El registro generado describe lo observado, sin atribuir resultados a retrieval.
El principal límite de alcance es documental: autorización pendiente para el manual
actual y separación obligatoria del antecedente histórico.

Este protocolo es útil antes de comparar arquitecturas porque evita confundir
procedencia, extracción y generación. El siguiente cuaderno estudiará qué estructura
del PDF sobrevive al parsing y qué páginas exigen revisión. No se concluye que una
arquitectura sea mejor hasta ejecutar experimentos comparables.

Lecturas: [Lewis et al.](https://arxiv.org/abs/2005.11401),
[E5](https://arxiv.org/abs/2402.05672),
[Qwen3 Embedding](https://arxiv.org/abs/2506.05176).
El roadmap y las decisiones se amplían en `docs/roadmap.md` y `docs/architecture.md`.